In [1]:
import pandas as pd
import lxml
import requests
import yfinance as yf
from io import StringIO

In [2]:
url= "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers= {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

In [3]:
response=requests.get(url, headers=headers)

sp500=pd.read_html(StringIO(response.text))[0]
sp500

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [4]:
sp500['Symbol']=sp500['Symbol'].str.replace(".", "-", regex=False)

In [5]:
# list of ticker symbols for download (Needs to be done for S&P 500)
tickerStrings = sp500['Symbol'].tolist()

In [ ]:
df = yf.download(tickerStrings, group_by='Ticker', period='1d')

[*****                 11%                       ]  54 of 503 completed$CDW: possibly delisted; no price data found  (period=1d)
[**********************55%*                      ]  279 of 503 completed

In [ ]:
df

In [ ]:
df = df.stack(level=0).rename_axis(['Date', 'Ticker']).reset_index()
df

In [ ]:
downloaded_tickers=set(df["Ticker"].unique())

In [ ]:
missing = set(tickerStrings) - downloaded_tickers

missing

In [ ]:
len(downloaded_tickers)

In [ ]:
row_counts = df.groupby("Ticker").size()
print(row_counts.sort_values())

In [ ]:
for ticker in ["FRT", "PHM", "SYY"]:
    count = (df["Ticker"] == ticker).sum()
    print(f"{ticker}: {count} rows")

In [ ]:
df.columns.name = None
df

In [ ]:
ohlcv_cols=["Open", "High", "Low", "Close", "Volume"]

In [ ]:
downloaded_tickers = set(df["Ticker"].unique())

downloaded_tickers

In [ ]:
fully_missing = set(expected_tickers) - downloaded_tickers

fully_missing

In [ ]:
def validate_download(df, expected_tickers, ohlcv_cols=["Open", "High", "Low", "Close", "Volume"]):
    
    downloaded_tickers = set(df["Ticker"].unique())
    fully_missing = set(expected_tickers) - downloaded_tickers

    # Per-column null counts per ticker
    null_by_col = (
        df.groupby("Ticker")[ohlcv_cols]
        .apply(lambda x: x.isna().sum())
    )
    null_by_col["Total"] = null_by_col.sum(axis=1)
    has_nulls = null_by_col[null_by_col["Total"] > 0]

    # First and last valid (non-null) date per ticker
    valid_dates = (
        df[df["Close"].notna()]
        .groupby("Ticker")["Date"]
        .agg(
            first_valid_date="min",
            last_valid_date="max"
        )
    )

    # Get the earliest date in your entire dataset (start of your backfill window)
    backfill_start = df["Date"].min()
    
    # Count trading days between backfill start and first valid date per ticker
    # Trading days = number of dates that appear in your df for any ticker
    all_trading_days = df["Date"].drop_duplicates().sort_values()
    
    def count_trading_days_before(first_valid_date):
        return (all_trading_days < first_valid_date).sum()
    
    valid_dates["trading_days_before_listing"] = valid_dates["first_valid_date"].apply(
        count_trading_days_before
    )
    # Distinguish fully null vs partially null
    row_counts = df.groupby("Ticker").size()
    max_possible_nulls = row_counts * len(ohlcv_cols)
    fully_null_mask = has_nulls["Total"] == max_possible_nulls[has_nulls.index]
    fully_null = has_nulls[fully_null_mask]
    partially_null = has_nulls[~fully_null_mask].sort_values("Total", ascending=False)

    # Join valid date info onto partially null summary
    partially_null_summary = partially_null.join(valid_dates, how="left")

    # Report
    print(f"Total tickers expected:  {len(expected_tickers)}")
    print(f"Total tickers downloaded: {len(downloaded_tickers)}")

    if fully_missing:
        print(f"\nFully missing (not in df at all): {fully_missing}")
    else:
        print("\nFully missing: none")

    if len(fully_null) > 0:
        print(f"\nPresent but completely null (failed download):")
        print(fully_null.to_string())
    else:
        print("\nCompletely null tickers: none")

    if len(partially_null_summary) > 0:
        print(f"\nPartially null (recent IPO/spin-off) — breakdown by column + valid date range:\n")
        print(partially_null_summary.to_string())
    else:
        print("\nPartially null tickers: none")

    return fully_missing, fully_null, partially_null_summary

In [ ]:
fully_missing, fully_null, partially_null = validate_download(df, tickerStrings)

In [ ]:
df.to_csv('daily_data_2306.csv')

In [ ]:
ohlcv_cols=["Open", "High", "Low", "Close", "Volume"]

In [ ]:
null_counts_by_ticker = (
    df.groupby("Ticker")[ohlcv_cols]
    .apply(lambda x: x.isna().sum().sum())  # total nulls across all OHLCV cols, per ticker
    .sort_values(ascending=False)
)

print(null_counts_by_ticker.head(10))

In [ ]:
yf.download("GOOG", group_by='Ticker', period='1d')